<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day6_3_(260611)_JPA_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
%%writefile /content/jpa-context/jpa-context-demo/build.gradle

plugins {
    id 'java'
    id 'org.springframework.boot' version '3.3.5'
    id 'io.spring.dependency-management' version '1.1.6'
}

group = 'com.example'
version = '0.0.1-SNAPSHOT'

java {
    toolchain {
        languageVersion = JavaLanguageVersion.of(17)
    }
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    implementation 'org.springframework.boot:spring-boot-starter-data-jpa'

    runtimeOnly 'com.h2database:h2'

    testImplementation 'org.springframework.boot:spring-boot-starter-test'
}

tasks.named('test') {
    useJUnitPlatform()
}

Overwriting /content/jpa-context/jpa-context-demo/build.gradle


In [11]:
%%writefile /content/jpa-context/jpa-context-demo/src/main/resources/application.properties

server.port=3100
spring.application.name=jpa-context-demo

spring.datasource.url=jdbc:h2:mem:jpa_context_db
spring.datasource.driver-class-name=org.h2.Driver
spring.datasource.username=sa
spring.datasource.password=

spring.jpa.hibernate.ddl-auto=create
spring.jpa.show-sql=true
spring.jpa.properties.hibernate.format_sql=true
spring.jpa.properties.hibernate.highlight_sql=true

logging.level.org.hibernate.SQL=debug
logging.level.org.hibernate.orm.jdbc.bind=trace

Overwriting /content/jpa-context/jpa-context-demo/src/main/resources/application.properties


In [12]:
%%writefile /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/JpaContextDemoApplication.java

package com.example.demo;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class JpaContextDemoApplication {

    public static void main(String[] args) {
        SpringApplication.run(JpaContextDemoApplication.class, args);
    }
}


Overwriting /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/JpaContextDemoApplication.java


In [13]:
%%writefile /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/member/Member.java

package com.example.demo.member;

import jakarta.persistence.Entity;
import jakarta.persistence.GeneratedValue;
import jakarta.persistence.GenerationType;
import jakarta.persistence.Id;

@Entity
public class Member {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private String email;

    private String name;

    protected Member() {
    }

    public Member(String email, String name) {
        this.email = email;
        this.name = name;
    }

    public void changeName(String name) {
        this.name = name;
    }

    public Long getId() {
        return id;
    }

    public String getEmail() {
        return email;
    }

    public String getName() {
        return name;
    }
}

Overwriting /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/member/Member.java


In [14]:
%%writefile /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/member/MemberRepository.java

package com.example.demo.member;

import org.springframework.data.jpa.repository.JpaRepository;

public interface MemberRepository extends JpaRepository<Member, Long> {
}

Overwriting /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/member/MemberRepository.java


In [15]:
%%writefile /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/dto/MemberCreateRequest.java

package com.example.demo.dto;

public class MemberCreateRequest {

    private String email;
    private String name;

    public MemberCreateRequest() {
    }

    public MemberCreateRequest(String email, String name) {
        this.email = email;
        this.name = name;
    }

    public String getEmail() {
        return email;
    }

    public String getName() {
        return name;
    }

    public void setEmail(String email) {
        this.email = email;
    }

    public void setName(String name) {
        this.name = name;
    }
}

Overwriting /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/dto/MemberCreateRequest.java


In [16]:
%%writefile /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/dto/MemberResponse.java

package com.example.demo.dto;

public class MemberResponse {

    private Long id;
    private String email;
    private String name;

    public MemberResponse(Long id, String email, String name) {
        this.id = id;
        this.email = email;
        this.name = name;
    }

    public Long getId() {
        return id;
    }

    public String getEmail() {
        return email;
    }

    public String getName() {
        return name;
    }
}

Overwriting /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/dto/MemberResponse.java


In [17]:
%%writefile /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/member/MemberService.java

package com.example.demo.member;

import com.example.demo.dto.MemberCreateRequest;
import com.example.demo.dto.MemberResponse;
import jakarta.persistence.EntityManager;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;

@Service
public class MemberService {

    private final MemberRepository memberRepository;
    private final EntityManager entityManager;

    public MemberService(MemberRepository memberRepository, EntityManager entityManager) {
        this.memberRepository = memberRepository;
        this.entityManager = entityManager;
    }

    @Transactional
    public MemberResponse create(MemberCreateRequest request) {
        System.out.println("1. Member 객체 생성: 비영속 상태");

        Member member = new Member(request.getEmail(), request.getName());

        System.out.println("2. save() 호출 전");
        Member savedMember = memberRepository.save(member);
        System.out.println("3. save() 호출 후");
        System.out.println("4. 아직 트랜잭션은 끝나지 않음");

        return new MemberResponse(
                savedMember.getId(),
                savedMember.getEmail(),
                savedMember.getName()
        );
    }

    @Transactional(readOnly = true)
    public MemberResponse find(Long id) {
        System.out.println("1. 첫 번째 findById 호출");
        Member member1 = memberRepository.findById(id)
                .orElseThrow(() -> new IllegalArgumentException("회원을 찾을 수 없습니다."));

        System.out.println("2. 두 번째 findById 호출");
        Member member2 = memberRepository.findById(id)
                .orElseThrow(() -> new IllegalArgumentException("회원을 찾을 수 없습니다."));

        System.out.println("3. member1 == member2 결과: " + (member1 == member2));

        return new MemberResponse(
                member1.getId(),
                member1.getEmail(),
                member1.getName()
        );
    }

    @Transactional
    public MemberResponse changeName(Long id, String name) {
        System.out.println("1. findById 호출: 엔티티 조회");

        Member member = memberRepository.findById(id)
                .orElseThrow(() -> new IllegalArgumentException("회원을 찾을 수 없습니다."));

        System.out.println("2. 엔티티 이름 변경 전: " + member.getName());

        member.changeName(name);

        System.out.println("3. 엔티티 이름 변경 후: " + member.getName());
        System.out.println("4. save()를 다시 호출하지 않음");
        System.out.println("5. 트랜잭션 종료 시 변경 감지로 UPDATE SQL 실행 가능");

        return new MemberResponse(
                member.getId(),
                member.getEmail(),
                member.getName()
        );
    }

    @Transactional
    public String flushTest() {
        System.out.println("1. Member 객체 생성");

        Member member = new Member("flush@test.com", "flush-user");

        System.out.println("2. save() 호출");
        memberRepository.save(member);

        System.out.println("3. flush() 호출 전");
        entityManager.flush();
        System.out.println("4. flush() 호출 후");

        return "flush 테스트 완료";
    }
}

Overwriting /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/member/MemberService.java


In [18]:
%%writefile /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/member/MemberController.java

package com.example.demo.member;

import com.example.demo.dto.MemberCreateRequest;
import com.example.demo.dto.MemberResponse;
import org.springframework.web.bind.annotation.*;

@RestController
@RequestMapping("/api/members")
public class MemberController {

    private final MemberService memberService;

    public MemberController(MemberService memberService) {
        this.memberService = memberService;
    }

    @PostMapping
    public MemberResponse create(@RequestBody MemberCreateRequest request) {
        return memberService.create(request);
    }

    @GetMapping("/{id}")
    public MemberResponse find(@PathVariable Long id) {
        return memberService.find(id);
    }

    @PatchMapping("/{id}/name")
    public MemberResponse changeName(
            @PathVariable Long id,
            @RequestParam String name
    ) {
        return memberService.changeName(id, name);
    }

    @PostMapping("/flush-test")
    public String flushTest() {
        return memberService.flushTest();
    }
}

Overwriting /content/jpa-context/jpa-context-demo/src/main/java/com/example/demo/member/MemberController.java
